## 1. Install Libraries

In [1]:
!pip install -q -U transformers accelerate sentencepiece safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 49.3 MB/s eta 0:00:00


## 2. Imports

In [2]:
import os
import gc
import json
import time
import warnings
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch

from transformers import (

    AutoTokenizer,

    AutoModelForSequenceClassification

)

warnings.filterwarnings("ignore")

## 3. Check GPU

In [3]:
print("="*70)
print("Hardware")
print("="*70)

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print("Device :", DEVICE)

if torch.cuda.is_available():

    print(

        "GPU :",

        torch.cuda.get_device_name(0)

    )

    print(

        "GPU Memory :",

        round(

            torch.cuda.get_device_properties(0).total_memory

            /1024**3,

            2

        ),

        "GB"

    )

Hardware
Device : cuda
GPU : Tesla T4
GPU Memory : 14.56 GB


## 4. Project Paths

In [4]:
PROJECT_PATH = "/content/drive/MyDrive/FinSight_AI"

DATA_PATH = os.path.join(

    PROJECT_PATH,

    "data"

)

PROCESSED_PATH = os.path.join(

    DATA_PATH,

    "processed"

)

CLASSIFICATION_PATH = os.path.join(

    DATA_PATH,

    "classification"

)

RULE_OUTPUT_PATH = os.path.join(

    CLASSIFICATION_PATH,

    "rule_classification"

)

SEMANTIC_PATH = os.path.join(

    CLASSIFICATION_PATH,

    "semantic_classification"

)

os.makedirs(

    SEMANTIC_PATH,

    exist_ok=True

)

## 5. Load Unresolved Headlines

In [5]:
UNRESOLVED_FILE = os.path.join(

    RULE_OUTPUT_PATH,

    "unresolved_news.parquet"

)

semantic_news = pd.read_parquet(

    UNRESOLVED_FILE

)

print("="*70)

print("Semantic Dataset")

print("="*70)

print(semantic_news.shape)

semantic_news.head()

Semantic Dataset
(1584926, 8)


,news_id,headline,event,market_signal,confidence,matched_keywords,rule_score,method
0,18,"Q1 13F Roundup: How Buffett, Einhorn, Ackman A...",Other,Neutral,0.0,,0.0,Rule
1,19,Pershing Square 13F Shows Fund Raises Stake In...,Other,Neutral,0.0,,0.0,Rule
2,20,How Bill Ackman Successfully Navigated Coronav...,Other,Neutral,0.0,,0.0,Rule
3,25,Agilent Reports Has Become Top-Level Sponsor O...,Other,Neutral,0.0,,0.0,Rule
4,54,"Q4 13F Roundup: How Buffett, Einhorn, Ackman A...",Other,Neutral,0.0,,0.0,Rule


## 6. Event Labels

In [6]:
EVENT_LABELS = [

    "Earnings",

    "Revenue Guidance",

    "Analyst Rating",

    "Dividend",

    "Stock Split",

    "Share Buyback",

    "Merger & Acquisition",

    "Partnership",

    "IPO",

    "Funding",

    "Executive Change",

    "Layoffs",

    "Product Launch",

    "Regulatory Approval",

    "Litigation",

    "Cybersecurity",

    "Fraud",

    "Bankruptcy",

    "Credit Rating",

    "Monetary Policy",

    "Economic Data",

    "Commodity",

    "Cryptocurrency",

    "ESG",

    "Market Movement",

    "Technical Analysis",

    "Options Activity",

    "Trading Halt",

    "Insider Trading"

]

print(len(EVENT_LABELS))

29


## 7. Load Sentence Transformer

In [7]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-large-en-v1.5"

embedding_model = SentenceTransformer(

    MODEL_NAME,

    device=DEVICE

)

print("="*70)

print("Sentence Transformer Loaded")

print("="*70)

print(MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Sentence Transformer Loaded
BAAI/bge-large-en-v1.5


## 8. Financial Event Descriptions

In [8]:
EVENT_DESCRIPTIONS = {
    "Earnings": """This financial news is about a company reporting quarterly or annual earnings, earnings per share (EPS), revenue, profits, financial performance, quarterly or annual financial results, or issuing future financial guidance.

Representative examples:
- Apple reports record quarterly earnings and raises guidance.
- Microsoft beats EPS estimates for the fourth quarter.
- Amazon reports stronger-than-expected revenue growth.
""",

    "Revenue Guidance": """This financial news is about a company increasing, lowering, reaffirming or updating future revenue, earnings, profit guidance, forecasts or business outlook.

Representative examples:
- Tesla raises full-year revenue guidance.
- Intel lowers its quarterly outlook.
- Adobe reaffirms annual guidance.
""",

    "Analyst Rating": """This financial news is about analysts or brokerages upgrading, downgrading, maintaining ratings, or changing price targets.

Representative examples:
- Goldman Sachs upgrades NVIDIA to Buy.
- Barclays maintains Equal-Weight on Apple.
- Morgan Stanley raises Tesla price target.
""",

    "Dividend": """This financial news is about dividend declarations, payments, increases, decreases or special dividends.

Representative examples:
- Coca-Cola declares quarterly dividend.
- Microsoft raises annual dividend.
- IBM announces special dividend.
""",

    "Stock Split": """This financial news is about stock splits or reverse stock splits.

Representative examples:
- Tesla announces a stock split.
- NVIDIA completes a stock split.
- Company approves reverse stock split.
""",

    "Share Buyback": """This financial news is about share repurchases or stock buyback programs.

Representative examples:
- Apple expands buyback program.
- Meta authorizes share repurchase.
- Company completes stock buyback.
""",

    "Merger & Acquisition": """This financial news is about mergers, acquisitions, takeovers or corporate buyouts.

Representative examples:
- Microsoft acquires Activision Blizzard.
- Broadcom completes VMware acquisition.
- Company announces merger agreement.
""",

    "Partnership": """This financial news is about strategic partnerships, collaborations, alliances or joint ventures.

Representative examples:
- NVIDIA partners with Microsoft.
- Pfizer signs collaboration agreement.
- Amazon enters strategic alliance.
""",

    "IPO": """This financial news is about initial public offerings, direct listings or public listings.

Representative examples:
- Company files for IPO.
- Startup begins NYSE listing.
- Firm prices initial public offering.
""",

    "Funding": """This financial news is about venture capital funding, investment rounds, fundraising or capital raising.

Representative examples:
- Startup raises Series B funding.
- Company secures venture capital investment.
- Firm closes funding round.
""",

    "Executive Change": """This financial news is about executive appointments, resignations, retirements or leadership changes.

Representative examples:
- Apple appoints new CFO.
- CEO resigns unexpectedly.
- Company names new chairman.
""",

    "Layoffs": """This financial news is about layoffs, workforce reductions, restructuring or job cuts.

Representative examples:
- Intel cuts 5000 jobs.
- Company announces layoffs.
- Firm begins restructuring.
""",

    "Product Launch": """This financial news is about launching or releasing new products, services or technologies.

Representative examples:
- Apple unveils new iPhone.
- Google launches AI platform.
- Tesla introduces new vehicle.
""",

    "Regulatory Approval": """This financial news is about FDA approvals, SEC approvals, regulatory clearance or official authorization.

Representative examples:
- Pfizer receives FDA approval.
- Drug gains regulatory clearance.
- SEC approves ETF.
""",

    "Litigation": """This financial news is about lawsuits, settlements, investigations or legal disputes.

Representative examples:
- Company settles lawsuit.
- Court rules against firm.
- Legal action filed.
""",

    "Cybersecurity": """This financial news is about cyber attacks, data breaches, ransomware or hacking incidents.

Representative examples:
- Coinbase suffers data breach.
- Company hit by ransomware.
- Hack exposes customer information.
""",

    "Fraud": """This financial news is about fraud, accounting scandals, investigations or financial misconduct.

Representative examples:
- SEC investigates fraud.
- Executive charged with fraud.
- Company accused of misconduct.
""",

    "Bankruptcy": """This financial news is about bankruptcy, insolvency, liquidation or Chapter 11 filings.

Representative examples:
- Company files Chapter 11.
- Retailer enters bankruptcy.
- Business begins liquidation.
""",

    "Credit Rating": """This financial news is about Moody's, Fitch or S&P rating actions.

Representative examples:
- Moody's upgrades debt.
- Fitch downgrades credit rating.
- S&P revises outlook.
""",

    "Monetary Policy": """This financial news is about Federal Reserve decisions, central banks, interest rates or monetary policy.

Representative examples:
- Federal Reserve raises rates.
- ECB cuts rates.
- Central bank policy announcement.
""",

    "Economic Data": """This financial news is about CPI, GDP, payrolls, unemployment, inflation or macroeconomic indicators.

Representative examples:
- CPI inflation rises.
- GDP beats expectations.
- Payroll report released.
""",

    "Commodity": """This financial news is about oil, gold, silver, copper, natural gas or commodity markets.

Representative examples:
- Oil prices surge.
- Gold reaches record high.
- Natural gas declines.
""",

    "Cryptocurrency": """This financial news is about Bitcoin, Ethereum, blockchain, crypto exchanges or digital assets.

Representative examples:
- Bitcoin rallies.
- Ethereum upgrade announced.
- Coinbase expands crypto services.
""",

    "ESG": """This financial news is about environmental, social, governance or sustainability initiatives.

Representative examples:
- Company releases ESG report.
- Net-zero commitment announced.
- Renewable investment expands.
""",

    "Market Movement": """This financial news is about stock market movement, top gainers, top losers or unusual price changes.

Representative examples:
- Stocks hit 52-week highs.
- Biggest market movers today.
- Shares trade sharply lower.
""",

    "Technical Analysis": """This financial news is about chart patterns, support, resistance, breakouts or technical indicators.

Representative examples:
- Stock breaks resistance.
- Bullish breakout confirmed.
- Moving average crossover.
""",

    "Options Activity": """This financial news is about unusual options activity, call options, put options or option alerts.

Representative examples:
- Unusual options activity detected.
- Heavy call buying.
- Option alert issued.
""",

    "Trading Halt": """This financial news is about trading halts, exchange suspensions or volatility pauses.

Representative examples:
- NASDAQ halts trading.
- Stock resumes after halt.
- Exchange pauses trading.
""",

    "Insider Trading": """This financial news is about executives, directors or insiders buying or selling company shares.

Representative examples:
- CEO buys company stock.
- Director sells shares.
- Insider filing reported.
"""
}


## 9. Encode Event Descriptions

In [9]:
event_names = list(EVENT_DESCRIPTIONS.keys())

# event_texts = list(EVENT_DESCRIPTIONS.values())
event_texts = [

    "Represent this financial event category for semantic matching:\n\n" + description

    for description in EVENT_DESCRIPTIONS.values()

]

event_embeddings = embedding_model.encode(

    event_texts,

    batch_size=32,

    convert_to_tensor=True,

    normalize_embeddings=True,

    show_progress_bar=True

)

print("="*70)

print("Event Embeddings Created")

print("="*70)

print(f"Total Events : {len(event_names)}")

print(f"Embedding Shape : {event_embeddings.shape}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Event Embeddings Created
Total Events : 29
Embedding Shape : torch.Size([29, 1024])


## 10. Semantic Matching Function

In [10]:
# Semantic Classification Configuration
SIMILARITY_THRESHOLD = 0.45
TOP_K = 3
EMBEDDING_BATCH_SIZE = 128

In [13]:
from sentence_transformers import util

def semantic_classify(headlines):

    headline_embeddings = embedding_model.encode(

        [
            "Represent this financial news headline for semantic event classification:\n" + h
            for h in headlines
        ],

        batch_size=EMBEDDING_BATCH_SIZE,

        convert_to_tensor=True,

        normalize_embeddings=True,

        show_progress_bar=False

    )

    similarity = util.cos_sim(

        headline_embeddings,

        event_embeddings

    )

    scores, indices = torch.topk(

        similarity,

        k=TOP_K,

        dim=1

    )

    predictions = []

    for score_list, idx_list in zip(scores, indices):

        candidates = []

        for score, idx in zip(score_list, idx_list):

            score = score.item()

            if score >= SIMILARITY_THRESHOLD:

                event = event_names[idx.item()]

            else:

                event = "Unknown"

            candidates.append({

                "event": event,

                "confidence": round(score, 4)

            })

        best_prediction = candidates[0]

        predictions.append({

            "semantic_event": best_prediction["event"],

            "semantic_confidence": best_prediction["confidence"],

            "top_k_predictions": candidates

        })

    return predictions

## 11. Test on Sample Headlines

In [14]:
test_headlines = [

    "Apple beats quarterly earnings estimates",

    "Microsoft acquires AI startup",

    "Federal Reserve raises interest rates",

    "Tesla announces stock split",

    "Coinbase hacked",

    "Pfizer receives FDA approval",

    "Intel cuts 5000 jobs",

    "Company raises Series B funding",

    "Stocks hit new 52-week highs"

]

results = semantic_classify(

    test_headlines

)

for headline, result in zip(

    test_headlines,

    results

):

    print("="*80)

    print(headline)

    print(result)

Apple beats quarterly earnings estimates
{'semantic_event': 'Product Launch', 'semantic_confidence': 0.7653, 'top_k_predictions': [{'event': 'Product Launch', 'confidence': 0.7653}, {'event': 'Earnings', 'confidence': 0.7494}, {'event': 'Market Movement', 'confidence': 0.7294}]}
Microsoft acquires AI startup
{'semantic_event': 'Merger & Acquisition', 'semantic_confidence': 0.8099, 'top_k_predictions': [{'event': 'Merger & Acquisition', 'confidence': 0.8099}, {'event': 'Product Launch', 'confidence': 0.775}, {'event': 'Partnership', 'confidence': 0.7389}]}
Federal Reserve raises interest rates
{'semantic_event': 'Monetary Policy', 'semantic_confidence': 0.8093, 'top_k_predictions': [{'event': 'Monetary Policy', 'confidence': 0.8093}, {'event': 'Economic Data', 'confidence': 0.7445}, {'event': 'Product Launch', 'confidence': 0.7433}]}
Tesla announces stock split
{'semantic_event': 'Stock Split', 'semantic_confidence': 0.8239, 'top_k_predictions': [{'event': 'Stock Split', 'confidence': 0

## 13. Batch Configuration

In [15]:
BATCH_SIZE = 5000

TOTAL_ROWS = len(semantic_news)

TOTAL_BATCHES = (

    TOTAL_ROWS + BATCH_SIZE - 1

) // BATCH_SIZE

print("=" * 70)

print("Semantic Classification")

print("=" * 70)

print(f"Total Headlines : {TOTAL_ROWS:,}")

print(f"Batch Size      : {BATCH_SIZE:,}")

print(f"Total Batches   : {TOTAL_BATCHES}")

Semantic Classification
Total Headlines : 1,584,926
Batch Size      : 5,000
Total Batches   : 317


## 14. Output Paths

In [16]:
SEMANTIC_BATCH_PATH = os.path.join(

    SEMANTIC_PATH,

    "semantic_batches"

)

os.makedirs(

    SEMANTIC_BATCH_PATH,

    exist_ok=True

)

print(SEMANTIC_BATCH_PATH)

/content/drive/MyDrive/FinSight_AI/data/classification/semantic_classification/semantic_batches


##15. Checkpoint Functions

In [17]:
CHECKPOINT_FILE = os.path.join(

    SEMANTIC_PATH,

    "semantic_checkpoint.json"

)

def save_checkpoint(

    batch_number,

    processed_rows

):

    checkpoint = {

        "last_batch": batch_number,

        "processed_rows": processed_rows

    }

    with open(

        CHECKPOINT_FILE,

        "w"

    ) as f:

        json.dump(

            checkpoint,

            f,

            indent=4

        )


def load_checkpoint():

    if os.path.exists(

        CHECKPOINT_FILE

    ):

        with open(

            CHECKPOINT_FILE,

            "r"

        ) as f:

            return json.load(f)

    return {

        "last_batch": 0,

        "processed_rows": 0

    }

## 16. Process One Batch

In [18]:
def process_batch(batch_df):

    predictions = semantic_classify(

        batch_df["headline"].tolist()

    )

    output = batch_df[

        [

            "news_id",

            "headline"

        ]

    ].copy()

    output["semantic_event"] = [

        p["semantic_event"]

        for p in predictions

    ]

    output["semantic_confidence"] = [

        p["semantic_confidence"]

        for p in predictions

    ]

    output["top_k_predictions"] = [

        json.dumps(

            p["top_k_predictions"]

        )

        for p in predictions

    ]

    return output

## 17. Test One Batch

In [19]:
test_batch = semantic_news.iloc[:20]

test_output = process_batch(

    test_batch

)

display(test_output.head())

,news_id,headline,semantic_event,semantic_confidence,top_k_predictions
0,18,"Q1 13F Roundup: How Buffett, Einhorn, Ackman A...",Analyst Rating,0.7577,"[{""event"": ""Analyst Rating"", ""confidence"": 0.7..."
1,19,Pershing Square 13F Shows Fund Raises Stake In...,Funding,0.7031,"[{""event"": ""Funding"", ""confidence"": 0.7031}, {..."
2,20,How Bill Ackman Successfully Navigated Coronav...,Technical Analysis,0.7333,"[{""event"": ""Technical Analysis"", ""confidence"":..."
3,25,Agilent Reports Has Become Top-Level Sponsor O...,ESG,0.7724,"[{""event"": ""ESG"", ""confidence"": 0.7724}, {""eve..."
4,54,"Q4 13F Roundup: How Buffett, Einhorn, Ackman A...",Analyst Rating,0.7478,"[{""event"": ""Analyst Rating"", ""confidence"": 0.7..."


In [20]:
test_batch = semantic_news.iloc[:20]

test_output = process_batch(

    test_batch

)

display(test_output.head())

,news_id,headline,semantic_event,semantic_confidence,top_k_predictions
0,18,"Q1 13F Roundup: How Buffett, Einhorn, Ackman A...",Analyst Rating,0.7577,"[{""event"": ""Analyst Rating"", ""confidence"": 0.7..."
1,19,Pershing Square 13F Shows Fund Raises Stake In...,Funding,0.7031,"[{""event"": ""Funding"", ""confidence"": 0.7031}, {..."
2,20,How Bill Ackman Successfully Navigated Coronav...,Technical Analysis,0.7333,"[{""event"": ""Technical Analysis"", ""confidence"":..."
3,25,Agilent Reports Has Become Top-Level Sponsor O...,ESG,0.7724,"[{""event"": ""ESG"", ""confidence"": 0.7724}, {""eve..."
4,54,"Q4 13F Roundup: How Buffett, Einhorn, Ackman A...",Analyst Rating,0.7478,"[{""event"": ""Analyst Rating"", ""confidence"": 0.7..."


## 18. Load Checkpoint

In [21]:
checkpoint = load_checkpoint()

START_BATCH = checkpoint["last_batch"]

processed_rows = checkpoint["processed_rows"]

print("=" * 70)

print("Checkpoint Loaded")

print("=" * 70)

print(f"Start Batch    : {START_BATCH}")

print(f"Processed Rows : {processed_rows:,}")

Checkpoint Loaded
Start Batch    : 0
Processed Rows : 0


## 19. Process Entire Dataset

In [22]:
overall_start = time.time()

print("=" * 70)

print("Processing Semantic Classification")

print("=" * 70)

for batch_number in tqdm(

    range(

        START_BATCH,

        TOTAL_BATCHES

    ),

    desc="Semantic Classification"

):

    output_file = os.path.join(

        SEMANTIC_BATCH_PATH,

        f"semantic_batch_{batch_number+1:04d}.parquet"

    )

    if os.path.exists(output_file):

        continue

    start = batch_number * BATCH_SIZE

    end = min(

        start + BATCH_SIZE,

        TOTAL_ROWS

    )

    batch_df = semantic_news.iloc[start:end]

    batch_start = time.time()

    batch_result = process_batch(

        batch_df

    )

    batch_result.to_parquet(

        output_file,

        index=False

    )

    processed_rows += len(batch_result)

    save_checkpoint(

        batch_number + 1,

        processed_rows

    )

    elapsed = time.time() - overall_start

    avg_batch = elapsed / (batch_number + 1)

    eta = (

        TOTAL_BATCHES

        - batch_number

        - 1

    ) * avg_batch / 60

    if (

        (batch_number + 1) % 10 == 0

        or

        batch_number == TOTAL_BATCHES - 1

    ):

        print()

        print("=" * 70)

        print(f"Batch : {batch_number+1}/{TOTAL_BATCHES}")

        print(f"Rows Processed : {processed_rows:,}")

        print(f"ETA : {eta:.2f} minutes")

        print(f"Batch Time : {time.time()-batch_start:.2f} sec")

        print(f"Average Confidence : {batch_result['semantic_confidence'].mean():.4f}")

    del batch_df
    del batch_result

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

print()

print("=" * 70)

print("SEMANTIC CLASSIFICATION COMPLETED")

print("=" * 70)

print(f"Processed Rows : {processed_rows:,}")

Processing Semantic Classification


Semantic Classification:   0%|          | 0/317 [00:00<?, ?it/s]


Batch : 10/317
Rows Processed : 50,000
ETA : 228.13 minutes
Batch Time : 42.90 sec
Average Confidence : 0.7370

Batch : 20/317
Rows Processed : 100,000
ETA : 219.74 minutes
Batch Time : 42.68 sec
Average Confidence : 0.7333

Batch : 30/317
Rows Processed : 150,000
ETA : 211.36 minutes
Batch Time : 43.15 sec
Average Confidence : 0.7296

Batch : 40/317
Rows Processed : 200,000
ETA : 208.16 minutes
Batch Time : 44.85 sec
Average Confidence : 0.7272

Batch : 50/317
Rows Processed : 250,000
ETA : 199.99 minutes
Batch Time : 44.87 sec
Average Confidence : 0.7291

Batch : 60/317
Rows Processed : 300,000
ETA : 192.14 minutes
Batch Time : 44.13 sec
Average Confidence : 0.7344

Batch : 70/317
Rows Processed : 350,000
ETA : 184.72 minutes
Batch Time : 47.45 sec
Average Confidence : 0.7259

Batch : 80/317
Rows Processed : 400,000
ETA : 177.02 minutes
Batch Time : 44.21 sec
Average Confidence : 0.7294

Batch : 90/317
Rows Processed : 450,000
ETA : 169.52 minutes
Batch Time : 44.61 sec
Average Conf

## 20. Merge All Batch Files

In [23]:
semantic_files = sorted(

    os.listdir(

        SEMANTIC_BATCH_PATH

    )

)

semantic_df = pd.concat(

    (

        pd.read_parquet(

            os.path.join(

                SEMANTIC_BATCH_PATH,

                file

            )

        )

        for file in tqdm(

            semantic_files,

            desc="Merging Semantic Batches"

        )

    ),

    ignore_index=True

)

OUTPUT_FILE = os.path.join(

    SEMANTIC_PATH,

    "semantic_classification.parquet"

)

semantic_df.to_parquet(

    OUTPUT_FILE,

    index=False

)

print()

print("=" * 70)

print("MASTER SEMANTIC FILE CREATED")

print("=" * 70)

print(semantic_df.shape)

semantic_df.head()

Merging Semantic Batches:   0%|          | 0/317 [00:00<?, ?it/s]


MASTER SEMANTIC FILE CREATED
(1584926, 5)


,news_id,headline,semantic_event,semantic_confidence,top_k_predictions
0,18,"Q1 13F Roundup: How Buffett, Einhorn, Ackman A...",Analyst Rating,0.7577,"[{""event"": ""Analyst Rating"", ""confidence"": 0.7..."
1,19,Pershing Square 13F Shows Fund Raises Stake In...,Funding,0.7031,"[{""event"": ""Funding"", ""confidence"": 0.7031}, {..."
2,20,How Bill Ackman Successfully Navigated Coronav...,Technical Analysis,0.7333,"[{""event"": ""Technical Analysis"", ""confidence"":..."
3,25,Agilent Reports Has Become Top-Level Sponsor O...,ESG,0.7724,"[{""event"": ""ESG"", ""confidence"": 0.7724}, {""eve..."
4,54,"Q4 13F Roundup: How Buffett, Einhorn, Ackman A...",Analyst Rating,0.7478,"[{""event"": ""Analyst Rating"", ""confidence"": 0.7..."


## 21. Evaluation

In [24]:
semantic_summary = (

    semantic_df

    .groupby("semantic_event")

    .agg(

        headlines=("news_id","count"),

        avg_confidence=("semantic_confidence","mean")

    )

    .sort_values(

        "headlines",

        ascending=False

    )

    .reset_index()

)

display(semantic_summary)

,semantic_event,headlines,avg_confidence
0,Product Launch,635242,0.740457
1,Market Movement,296058,0.736682
2,Merger & Acquisition,190725,0.741352
3,Commodity,103399,0.732615
4,Technical Analysis,100031,0.731294
5,Analyst Rating,52931,0.713177
6,ESG,38431,0.729800
7,Layoffs,34739,0.733357
8,Options Activity,18528,0.730398
9,Cryptocurrency,13511,0.722641


## 22. Summary

In [25]:
summary = pd.DataFrame({

    "Metric":[

        "Rows",

        "Semantic Events",

        "Batch Files",

        "Embedding Model",

        "Output File"

    ],

    "Value":[

        len(semantic_df),

        semantic_df["semantic_event"].nunique(),

        len(semantic_files),

        MODEL_NAME,

        os.path.basename(

            OUTPUT_FILE

        )

    ]

})

display(summary)

,Metric,Value
0,Rows,1584926
1,Semantic Events,29
2,Batch Files,317
3,Embedding Model,BAAI/bge-large-en-v1.5
4,Output File,semantic_classification.parquet
